## Setup and Data Loading

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Load the MSR 2026 dataset
print("Loading MSR 2026 AI Development dataset...")
df = pd.read_csv('../data/raw/aidata.csv')

print(f"Dataset loaded successfully!")
print(f"Total records: {len(df):,}")
print(f"Unique users: {df['user_id'].nunique():,}")
print(f"Unique agents: {df['agent'].nunique()}")
print(f"Dataset columns: {list(df.columns)}")

# Display first few rows
print("\nFirst 3 rows of the dataset:")
df.head(3)

## Stage 1: Raw Dataset Analysis and Concentration Metrics

In [ ]:
def compute_gini_coefficient(values):
    """Compute Gini coefficient for inequality measurement"""
    values = np.array(values)
    values = np.sort(values)
    n = len(values)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * values)) / (n * np.sum(values)) - (n + 1) / n

def compute_shannon_entropy(series):
    """Compute Shannon entropy for diversity measurement"""
    value_counts = series.value_counts()
    probabilities = value_counts / len(series)
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

print("Statistical functions defined successfully")

In [ ]:
# Raw agent distribution analysis
print("=" * 60)
print("STAGE 1: RAW DATASET CHARACTERISTICS")
print("=" * 60)

# Agent distribution
agent_counts = df['agent'].value_counts()
total_records = len(df)

print("\nRaw Agent Distribution:")
for agent, count in agent_counts.items():
    pct = (count / total_records) * 100
    print(f"  {agent}: {count:,} PRs ({pct:.1f}%)")

# User contribution analysis
user_contributions = df['user_id'].value_counts()
unique_users = len(user_contributions)
mean_contrib = user_contributions.mean()
median_contrib = user_contributions.median()

print(f"\nUser Distribution Characteristics:")
print(f"  Total unique users: {unique_users:,} accounts")
print(f"  Mean contributions: {mean_contrib:.1f} pull requests per user")
print(f"  Median contributions: {median_contrib:.1f} pull requests per user")
print(f"  Mean/Median ratio: {mean_contrib/median_contrib:.1f}x")

# Store raw results
raw_results = {
    'agent_distribution': dict(agent_counts),
    'agent_percentages': {agent: (count/total_records)*100 for agent, count in agent_counts.items()},
    'total_records': total_records,
    'unique_users': unique_users,
    'mean_contrib': mean_contrib,
    'median_contrib': median_contrib
}

In [ ]:
# Concentration metrics computation
print("\nConcentration Metrics:")

# Gini coefficient
gini = compute_gini_coefficient(user_contributions.values)
print(f"  Gini coefficient: {gini:.3f} (high inequality)")

# Shannon entropy for agents
agent_entropy = compute_shannon_entropy(df['agent'])
max_agent_entropy = np.log2(len(agent_counts))
normalized_entropy = agent_entropy / max_agent_entropy
print(f"  Shannon entropy (agents): {agent_entropy:.3f}")
print(f"  Normalized entropy: {normalized_entropy:.3f} ({normalized_entropy*100:.1f}% of maximum)")

# Top percentile analysis
top_1_pct_count = int(unique_users * 0.01)
top_1_pct_users = user_contributions.head(top_1_pct_count)
top_1_pct_share = (top_1_pct_users.sum() / total_records) * 100

top_01_pct_count = int(unique_users * 0.001)
top_01_pct_users = user_contributions.head(top_01_pct_count)
top_01_pct_share = (top_01_pct_users.sum() / total_records) * 100

# 99th percentile threshold
percentile_99 = user_contributions.quantile(0.99)

print(f"  Top-1% users ({top_1_pct_count:,}): {top_1_pct_share:.1f}% of total PRs")
print(f"  Top-0.1% users ({top_01_pct_count:,}): {top_01_pct_share:.1f}% of total PRs")
print(f"  99th percentile threshold: {percentile_99:.0f} PRs per user")

# Update results
raw_results.update({
    'gini_coefficient': gini,
    'shannon_entropy': agent_entropy,
    'normalized_entropy': normalized_entropy,
    'top_1_pct_share': top_1_pct_share,
    'top_01_pct_share': top_01_pct_share,
    'percentile_99_threshold': percentile_99
})

## Stage 2: Statistical Outlier Filtering (99th Percentile)

In [ ]:
print("=" * 60)
print("STAGE 2: STATISTICAL OUTLIER FILTERING")
print("=" * 60)

# Identify high-volume users (99th percentile and above)
high_volume_users = user_contributions[user_contributions >= percentile_99].index

# Create filtered dataset
df_filtered = df[~df['user_id'].isin(high_volume_users)].copy()

# Calculate filtering impact
accounts_removed = len(high_volume_users)
prs_removed = len(df) - len(df_filtered)
prs_removed_pct = (prs_removed / len(df)) * 100

# New user statistics
new_user_contributions = df_filtered['user_id'].value_counts()
new_mean_contrib = new_user_contributions.mean()

print(f"99th percentile threshold: {percentile_99:.0f} PRs per user")
print(f"Accounts removed: {accounts_removed:,} ({accounts_removed/unique_users*100:.1f}% of users)")
print(f"PRs removed: {prs_removed:,} ({prs_removed_pct:.1f}% of all PRs)")
print(f"New mean contribution: {new_mean_contrib:.1f} PRs per user")

# Agent distribution comparison
original_dist = df['agent'].value_counts(normalize=True) * 100
filtered_dist = df_filtered['agent'].value_counts(normalize=True) * 100

print(f"\nAgent Distribution Changes:")
filtering_changes = []
for agent in original_dist.index:
    orig_pct = original_dist[agent]
    filt_pct = filtered_dist.get(agent, 0)
    change = filt_pct - orig_pct
    print(f"  {agent}: {orig_pct:.1f}% -> {filt_pct:.1f}% ({change:+.1f}%)")
    
    filtering_changes.append({
        'agent': agent,
        'raw_pct': orig_pct,
        'filtered_pct': filt_pct,
        'delta': change
    })

# Store Stage 2 results
stage2_results = {
    'accounts_removed': accounts_removed,
    'prs_removed': prs_removed,
    'prs_removed_pct': prs_removed_pct,
    'new_mean_contrib': new_mean_contrib,
    'filtering_changes': filtering_changes,
    'original_distribution': dict(original_dist),
    'filtered_distribution': dict(filtered_dist)
}

## Stage 3: Bot Account Detection and Filtering

In [ ]:
def detect_bot_accounts(df):
    """Detect automated accounts using pattern matching"""
    print("Detecting bot accounts using pattern matching...")
    
    bot_patterns = [
        r'.*bot.*',           # Contains 'bot' (case insensitive)
        r'.*\[bot\].*',       # Contains '[bot]'
        r'.*-ci$',            # Ends with '-ci'
        r'.*-automation$',    # Ends with '-automation'
        r'.*dependabot.*',    # Dependabot variations
        r'.*renovate.*',      # Renovate bot
        r'.*github-actions.*', # GitHub Actions
        r'.*codecov.*',       # Codecov bot
        r'.*greenkeeper.*',   # Greenkeeper bot
    ]
    
    bot_users = set()
    pattern_counts = {}
    
    # Check usernames for bot patterns
    for pattern in bot_patterns:
        matches = df[df['user'].str.contains(pattern, case=False, na=False, regex=True)]
        if len(matches) > 0:
            pattern_users = matches['user_id'].unique()
            bot_users.update(pattern_users)
            pattern_counts[pattern] = len(pattern_users)
            print(f"  Pattern '{pattern}': {len(pattern_users)} users")
    
    return list(bot_users), pattern_counts

print("=" * 60)
print("STAGE 3: PATTERN-BASED BOT FILTERING")
print("=" * 60)

# Detect bot accounts in the filtered dataset
bot_user_ids, pattern_counts = detect_bot_accounts(df_filtered)

# Create final filtered dataset
df_final = df_filtered[~df_filtered['user_id'].isin(bot_user_ids)].copy()

# Calculate impact
bot_accounts = len(bot_user_ids)
bot_prs_removed = len(df_filtered) - len(df_final)

# Final agent distribution
final_dist = df_final['agent'].value_counts(normalize=True) * 100

print(f"\nBot Detection Summary:")
print(f"Total bot accounts detected: {bot_accounts:,}")
print(f"Additional PRs removed: {bot_prs_removed:,}")

print(f"\nFinal Debiased Agent Distribution:")
for agent, pct in final_dist.items():
    print(f"  {agent}: {pct:.1f}%")

# Store Stage 3 results
stage3_results = {
    'bot_accounts_detected': bot_accounts,
    'bot_prs_removed': bot_prs_removed,
    'pattern_counts': pattern_counts,
    'final_distribution': dict(final_dist)
}

# Overall filtering summary
total_retention_rate = len(df_final) / len(df)
print(f"\nOverall Filtering Summary:")
print(f"Original dataset: {len(df):,} records")
print(f"Final dataset: {len(df_final):,} records")
print(f"Retention rate: {total_retention_rate:.1%}")

## Visualization: Filtering Effects

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('MSR 2026 Filtering Study - Complete Analysis Results', fontsize=16, fontweight='bold')

# 1. Agent distribution comparison (before/after)
ax1 = axes[0, 0]
agents = list(original_dist.index)
raw_values = [original_dist[agent] for agent in agents]
filtered_values = [filtered_dist.get(agent, 0) for agent in agents]

x = np.arange(len(agents))
width = 0.35

ax1.bar(x - width/2, raw_values, width, label='Raw', alpha=0.8)
ax1.bar(x + width/2, filtered_values, width, label='Filtered', alpha=0.8)
ax1.set_xlabel('AI Agents')
ax1.set_ylabel('Percentage (%)')
ax1.set_title('Agent Distribution: Raw vs Filtered')
ax1.set_xticks(x)
ax1.set_xticklabels(agents, rotation=45, ha='right')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. User contribution distribution
ax2 = axes[0, 1]
contrib_bins = np.logspace(0, np.log10(user_contributions.max()), 50)
ax2.hist(user_contributions.values, bins=contrib_bins, alpha=0.7, edgecolor='black')
ax2.axvline(percentile_99, color='red', linestyle='--', linewidth=2, label=f'99th percentile ({percentile_99:.0f})')
ax2.set_xscale('log')
ax2.set_xlabel('Contributions per User (log scale)')
ax2.set_ylabel('Number of Users')
ax2.set_title('User Contribution Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Filtering impact summary
ax3 = axes[1, 0]
stages = ['Original', 'Stage 2\n(Outlier Filter)', 'Stage 3\n(Bot Filter)']
record_counts = [len(df), len(df_filtered), len(df_final)]
colors = ['blue', 'orange', 'green']

bars = ax3.bar(stages, record_counts, color=colors, alpha=0.7, edgecolor='black')
ax3.set_ylabel('Number of Records')
ax3.set_title('Dataset Size Through Filtering Stages')
ax3.grid(True, alpha=0.3)

# Add value labels on bars
for bar, count in zip(bars, record_counts):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{count:,}',
             ha='center', va='bottom', fontweight='bold')

# 4. Distribution changes table
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

# Create table data
table_data = []
for change in filtering_changes:
    table_data.append([
        change['agent'],
        f"{change['raw_pct']:.1f}%",
        f"{change['filtered_pct']:.1f}%",
        f"{change['delta']:+.1f}%"
    ])

table = ax4.table(cellText=table_data,
                  colLabels=['Agent', 'Raw %', 'Filtered %', 'Δ%'],
                  cellLoc='center',
                  loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 2)
ax4.set_title('Filtering Effects Table', pad=20)

plt.tight_layout()
plt.savefig('../outputs/figures/complete_filtering_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to: outputs/figures/complete_filtering_analysis.png")

## Export Complete Results

In [ ]:
# Compile all results
complete_results = {
    'study_metadata': {
        'title': 'User-Level Debiasing in AI Tool Adoption Studies',
        'author': 'Ahmed Mursal, Edinburgh Napier University',
        'analysis_date': datetime.now().isoformat(),
        'dataset_source': 'MSR 2026 Challenge - hao-li/AIDev from HuggingFace',
        'notebook_version': '1.0'
    },
    'stage1_raw_analysis': raw_results,
    'stage2_outlier_filtering': stage2_results,
    'stage3_bot_filtering': stage3_results,
    'summary_metrics': {
        'total_original_records': len(df),
        'total_final_records': len(df_final),
        'total_retention_rate': total_retention_rate,
        'total_accounts_removed': accounts_removed + bot_accounts,
        'total_prs_removed': prs_removed + bot_prs_removed,
        'mean_absolute_deviation': np.mean([abs(change['delta']) for change in filtering_changes])
    }
}

# Create outputs directory structure
output_dir = Path('../outputs/submission_ready')
output_dir.mkdir(exist_ok=True)
(output_dir / 'figures').mkdir(exist_ok=True)
(output_dir / 'data').mkdir(exist_ok=True)
(output_dir / 'scripts').mkdir(exist_ok=True)

# Save complete results
with open(output_dir / 'complete_analysis_results.json', 'w') as f:
    json.dump(complete_results, f, indent=2, default=str)

# Save filtered datasets
df_filtered.to_csv(output_dir / 'data' / 'stage2_filtered_dataset.csv', index=False)
df_final.to_csv(output_dir / 'data' / 'final_filtered_dataset.csv', index=False)

# Save summary statistics
summary_stats = {
    'paper_table_1': filtering_changes,
    'concentration_metrics': {
        'gini_coefficient': gini,
        'shannon_entropy': agent_entropy,
        'normalized_entropy': normalized_entropy
    },
    'filtering_impact': {
        'stage2_accounts_removed': accounts_removed,
        'stage3_bot_accounts': bot_accounts,
        'total_retention_rate': total_retention_rate
    }
}

with open(output_dir / 'paper_statistics.json', 'w') as f:
    json.dump(summary_stats, f, indent=2, default=str)

print("="*60)
print("ANALYSIS COMPLETE - ALL RESULTS EXPORTED")
print("="*60)
print(f"Results saved to: {output_dir}")
print(f"- complete_analysis_results.json: Full analysis results")
print(f"- paper_statistics.json: Statistics used in the paper")
print(f"- data/stage2_filtered_dataset.csv: Stage 2 filtered data")
print(f"- data/final_filtered_dataset.csv: Final filtered data")
print(f"- figures/complete_filtering_analysis.png: Main visualization")

## Reproducibility Verification

In [ ]:
# Verify key paper claims
print("REPRODUCIBILITY CHECK - PAPER CLAIMS VERIFICATION")
print("="*60)

paper_claims = {
    'Total records': 932791,
    'Unique users': 72189,
    'Gini coefficient': 0.829,
    'Shannon entropy': 0.769,
    'Top-1% share': 43.6,
    '99th percentile threshold': 178,
    'Accounts removed (Stage 2)': 725,
    'Bot accounts detected': 166
}

computed_values = {
    'Total records': len(df),
    'Unique users': unique_users,
    'Gini coefficient': round(gini, 3),
    'Shannon entropy': round(agent_entropy, 3),
    'Top-1% share': round(top_1_pct_share, 1),
    '99th percentile threshold': int(percentile_99),
    'Accounts removed (Stage 2)': accounts_removed,
    'Bot accounts detected': bot_accounts
}

all_verified = True
for claim, expected in paper_claims.items():
    actual = computed_values[claim]
    match = abs(actual - expected) < 0.01 if isinstance(expected, float) else actual == expected
    status = "✓ VERIFIED" if match else "✗ MISMATCH"
    print(f"{claim:<25}: {expected:<10} vs {actual:<10} {status}")
    if not match:
        all_verified = False

print("\n" + "="*60)
if all_verified:
    print("🎉 ALL PAPER CLAIMS SUCCESSFULLY VERIFIED!")
    print("The analysis is fully reproducible and academically sound.")
else:
    print("⚠️  Some values do not match paper claims.")
print("="*60)